# Notebook 04 — Ablation Study

The 4-configuration ablation table comparing CER and WER on 200 test samples:

| # | Configuration | CER | WER |
|---|---------------|-----|-----|
| 1 | Baseline TrOCR (no fine-tuning) | — | — |
| 2 | + Preprocessing (Module 1) | — | — |
| 3 | + Layout Analysis (Modules 1+2) | — | — |
| 4 | + Context Fusion (Modules 1+2+5) | — | — |

Run each configuration 3 times and report mean ± std.
Use the same 200 test samples for all configurations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jiwer
from PIL import Image
from pathlib import Path
import random
import json

from src.preprocessing import Preprocessor
from src.layout import LayoutAnalyser
from src.ocr import TrOCREngine
from src.candidates import build_candidate_sets
from src.context import ContextReasoner

print('Imports OK')

In [ ]:
# ── Load 200 test samples ─────────────────────────────────────────────────────
# (Same samples used for all 4 configurations)
IAM_DIR = Path('../data/iam')
# ... load all samples, pick fixed 200 test samples
# Ensure these 200 samples were NOT seen during training
test_200 = []  # populate
print(f'Test samples loaded: {len(test_200)}')

In [ ]:
# ── Configuration 1: Baseline TrOCR (no fine-tuning) ─────────────────────────
baseline_engine = TrOCREngine(model_path=None)  # loads microsoft/trocr-base-handwritten

def run_baseline(samples, engine):
    preds, refs = [], []
    for s in samples:
        img = Image.open(s['path'])
        cands = engine.recognise(img)
        preds.append(cands[0].word if cands else '')
        refs.append(s['label'])
    return jiwer.cer(refs, preds), jiwer.wer(refs, preds)

cer1, wer1 = run_baseline(test_200, baseline_engine)
print(f'Config 1 — Baseline: CER={cer1*100:.2f}%  WER={wer1*100:.2f}%')

In [ ]:
# ── Configuration 2: + Preprocessing ─────────────────────────────────────────
finetuned_engine = TrOCREngine(model_path='../models/trocr_finetuned')
pre = Preprocessor()

preds2, refs2 = [], []
for s in test_200:
    img = Image.open(s['path'])
    cleaned = pre.transform(img)
    cands = finetuned_engine.recognise(cleaned)
    preds2.append(cands[0].word if cands else '')
    refs2.append(s['label'])

cer2, wer2 = jiwer.cer(refs2, preds2), jiwer.wer(refs2, preds2)
print(f'Config 2 — + Preprocessing: CER={cer2*100:.2f}%  WER={wer2*100:.2f}%')

In [ ]:
# ── Configuration 3: + Layout ─────────────────────────────────────────────────
# Layout on word-level IAM samples is trivial (1 region = 1 word)
# This config is more meaningful on full-page documents.
# For IAM word samples: same as config 2, included for completeness.
cer3, wer3 = cer2, wer2  # approximately equal for word-level samples
print(f'Config 3 — + Layout: CER={cer3*100:.2f}%  WER={wer3*100:.2f}%')

In [ ]:
# ── Configuration 4: + Context Fusion ────────────────────────────────────────
reasoner = ContextReasoner(mode='anthropic')  # or 'ollama' or 'mock'

# For isolated word samples (IAM), context is limited.
# For full document pages, this provides the biggest boost.
# Run on full document test set for most meaningful comparison.

preds4, refs4 = [], []
# Process in groups of words to build context windows
# ... (implement windowed inference here using build_candidate_sets)
print('Config 4 — + Context Fusion: (implement windowed inference)')

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
results = [
    {'Configuration': 'Baseline TrOCR (no fine-tuning)',           'CER (%)': f'{cer1*100:.2f}', 'WER (%)': f'{wer1*100:.2f}'},
    {'Configuration': '+ Preprocessing (Module 1)',                 'CER (%)': f'{cer2*100:.2f}', 'WER (%)': f'{wer2*100:.2f}'},
    {'Configuration': '+ Layout Analysis (Modules 1+2)',            'CER (%)': f'{cer3*100:.2f}', 'WER (%)': f'{wer3*100:.2f}'},
    {'Configuration': '+ Context Fusion (Modules 1+2+5)',           'CER (%)': '—',               'WER (%)': '—'},
]
df = pd.DataFrame(results)
print(df.to_string(index=False))
df.to_csv('../outputs/ablation_table.csv', index=False)
print('\nSaved to outputs/ablation_table.csv')